# Numerical PDE Homework 1

In [1]:
import numpy as np
import scipy.linalg as la

import scipy.sparse as sp
import scipy.sparse.linalg as spla

import matplotlib as mpl
import matplotlib.pyplot as plt

import seaborn as sns
import seaborn.objects as so

# This setting makes it prints numerical outputs as, e.g. 3.14 and not np.float64(3.14)
# See https://numpy.org/doc/stable/reference/generated/numpy.set_printoptions.html
np.set_printoptions(legacy='1.25')

In [ ]:
# # Setup
x_start , x_end = 0.0 , 1.0
y_start , y_end = 0.0 , 1.0
mu = 1.0
Nx = 50
Ny = Nx

dx = ( x_end - x_start ) / Nx
dy = ( y_end - y_start ) / Ny

# # GRIDS - arrays of actual x and y values
# unknowns are ordered k = i + j* Nx (x varies fastest )

# p at cell centers
x_P = x_start + ( np . arange ( Nx ) + 0.5) * dx
y_P = y_start + ( np . arange ( Ny ) + 0.5) * dy

# u on east / west faces ( x = x_end is the same face as x = x_start )
x_U = x_start + np . arange ( Nx ) * dx
y_U = y_P

# v on north / south faces ( wall values are known , so they are removed )
x_V = x_P
y_V = y_start + np . arange (1 , Ny ) * dy

# Note the number of unknowns depends on boundary conditions !
nU = len(x_U) * len(y_U) #### # number of u unknowns
nV = len(x_V) * len(y_V) #### # number of v unknowns
nP = len(x_P) * len(y_P) #### # number of p unknowns

# Basic RHS functions

In [8]:
# Build the matrices in pieces

# U Laplacian

u_main_diag = -4 * np.ones(nU) # most common main diag value
u_main_diag[:Nx] -= 1
u_main_diag[-Nx:] -= 1 # This row and above come from BC U=0 at top and bottom of domain

u_super = np.ones(nU - 1)
u_sub = u_super.copy()

u_low = np.ones(nU - Ny)
u_high = np.ones(nU - Nx)

u_cols = np.concat((np.arange(Nx,nU), np.arange(1,nU), np.arange(nU), np.arange(nU-1), np.arange(nU-Ny)))
u_rows = np.concat((np.arange(nU-Nx), np.arange(nU-1), np.arange(nU), np.arange(1,nU), np.arange(Ny,nU)))
u_vals = np.concat((u_high, u_super, u_main_diag, u_sub, u_low))